# RAG Pipeline: MongoDB Local Atlas + Voyage AI (Local) + Gemma 4

**Session 2 | 50분 핸즈온 실습**

이 노트북 하나만으로 완전한 **RAG (Retrieval-Augmented Generation)** 파이프라인을 로컬 환경에서 처음부터 끝까지 구축합니다.
**외부 API 키가 전혀 필요 없습니다.** 임베딩 모델과 LLM 모두 로컬에서 실행됩니다.

| 구성 요소 | 도구 | 실행 위치 |
|---|---|---|
| 데이터베이스 | MongoDB Local Atlas (Docker) | 로컬 |
| 임베딩 모델 | `voyageai/voyage-4-nano` via SentenceTransformers | 로컬 |
| LLM | Ollama `gemma4:e4b` 기본, 8GB 메모리 장비는 `gemma4:e2b` | 로컬 |
| 메모리 | MongoDB `chat_history` 컬렉션 | 로컬 |


## 시작 전 확인

노트북을 실행하기 전에 **`hands-on/session-2/README.md`** 의 사전 준비 단계(Step 1~7)를 완료하세요.

| 확인 항목 | 명령어 |
|---|---|
| Docker 실행 중 | `docker ps` |
| Local Atlas 실행 중 | `atlas local list` |
| Ollama 실행 중 | `ollama list` |
| `.env` 파일 존재 | `cat work/.env` |
| uv 환경 생성됨 | `work/.venv` 디렉터리 존재 여부 확인 |

모두 확인했으면 아래 셀부터 순서대로 실행합니다.

## Step 1: 설정 및 연결 확인

환경 변수를 불러오고 MongoDB와 Ollama 연결을 확인합니다.


In [ ]:
import os
import json
import time
from datetime import datetime, timezone
from typing import List, Dict, Any

from dotenv import load_dotenv

load_dotenv()

# ── 연결 설정 ───────────────────────────────────────────────────────────────────
MONGODB_URI     = os.environ["MONGODB_URI"]
OLLAMA_BASE_URL = os.environ["OLLAMA_BASE_URL"]
OLLAMA_MODEL    = os.environ["OLLAMA_MODEL"]

# ── DB / 컬렉션 이름 ────────────────────────────────────────────────────────────
DB_NAME           = "rag_session2"
COLLECTION_NAME   = "knowledge_base"
CHAT_HISTORY_COLL = "chat_history"
VECTOR_INDEX_NAME = "vector_index"

# ── 임베딩 모델 ─────────────────────────────────────────────────────────────────
EMBEDDING_MODEL_ID = "voyageai/voyage-4-nano"

print("설정 완료:")
print(f"  MongoDB      : {MONGODB_URI}")
print(f"  Ollama 모델  : {OLLAMA_MODEL}")
print(f"  임베딩 모델  : {EMBEDDING_MODEL_ID} (로컬 실행)")

In [ ]:
from pymongo import MongoClient
from pymongo.operations import SearchIndexModel

mongo_client = MongoClient(MONGODB_URI)
mongo_client.admin.command("ping")  # 연결 실패 시 여기서 오류 발생

db             = mongo_client[DB_NAME]
collection     = db[COLLECTION_NAME]
history_coll   = db[CHAT_HISTORY_COLL]

print("MongoDB 연결 성공!")


## Step 2: 데이터 로드

MongoDB 공식 문서 스니펫 20개가 담긴 JSON 파일을 불러옵니다.
각 문서는 `title`, `body`, `url`, `metadata` 등의 필드를 가집니다.


In [ ]:
with open("../data/mongodb_docs.json") as f:
    docs = json.load(f)

print(f"문서 수: {len(docs)}개")
print()
print("── 첫 번째 문서 미리보기 ──")
d = docs[0]
print(f"title    : {d['title']}")
print(f"url      : {d['url']}")
print(f"updated  : {d['updated']}")
print(f"body[:200]: {d['body'][:200]}...")


## Step 3: 청킹 (Chunking)

LLM 컨텍스트 창에 맞게 긴 문서를 작은 조각으로 나눕니다.

**왜 청킹이 필요한가?**
- 임베딩 모델에는 입력 토큰 한도가 있습니다.
- 너무 큰 덩어리를 넣으면 검색 정밀도가 떨어집니다.
- 적절한 크기(800자 내외)로 나누면 관련 문단만 정확히 찾아낼 수 있습니다.

`RecursiveCharacterTextSplitter`는 문단(`\n\n`) → 줄(`\n`) → 단어 → 문자 순서로
자연스러운 경계를 우선하여 텍스트를 분할합니다.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm.notebook import tqdm

text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],
    chunk_size=800,    # 최대 청크 크기 (문자 수)
    chunk_overlap=80,  # 앞뒤 청크와 겹치는 문자 수 (문맥 연속성 유지)
)

def chunk_document(doc: Dict) -> List[Dict]:
    """문서의 body 필드를 청킹하고, 나머지 메타데이터는 그대로 유지."""
    chunks = text_splitter.split_text(doc["body"])
    result = []
    for chunk in chunks:
        chunk_doc = {k: v for k, v in doc.items() if k != "body"}
        chunk_doc["body"] = chunk
        result.append(chunk_doc)
    return result

all_chunks = []
for doc in tqdm(docs, desc="청킹 진행 중"):
    all_chunks.extend(chunk_document(doc))

print(f"원본 문서 수 : {len(docs)}개")
print(f"청킹 후 조각 수: {len(all_chunks)}개  (평균 {len(all_chunks)/len(docs):.1f}개/문서)")
print()
print("── 첫 번째 청크 미리보기 ──")
print(all_chunks[0]["body"][:300])


## Step 4: 임베딩 생성 (voyage-4-nano, 로컬)

각 텍스트 청크를 숫자 벡터(임베딩)로 변환합니다.
의미가 비슷한 텍스트는 벡터 공간에서 가까운 위치에 놓입니다.

사용 모델: **`voyageai/voyage-4-nano`** (HuggingFace, SentenceTransformers)
- Voyage AI의 최신 경량 임베딩 모델을 완전 로컬 실행
- API 키 불필요, 인터넷 연결은 최초 1회 다운로드 시에만 필요
- 이후 실행은 HuggingFace 캐시에서 불러옴

voyage-4-nano는 문서와 쿼리에 서로 다른 prefix를 내부적으로 추가하는 비대칭 임베딩 모델입니다.
반드시 용도에 맞는 메서드를 구분해 사용해야 검색 정확도가 높아집니다.
- `encode_document()`: 저장할 문서 청크에 사용
- `encode_query()`: 검색 시 사용자 질문에 사용

**처음 실행 시** 모델 다운로드로 1~2분이 소요될 수 있습니다.

In [ ]:
from sentence_transformers import SentenceTransformer

print(f"모델 로드 중: {EMBEDDING_MODEL_ID}")
print("(처음 실행 시 HuggingFace에서 다운로드, 이후 캐시 사용)")

# custom encode_query / encode_document를 사용하기 위해 trust_remote_code=True 설정
embed_model = SentenceTransformer(EMBEDDING_MODEL_ID, trust_remote_code=True)

# 차원 수 자동 감지
test_emb = embed_model.encode_query(["hello"])
EMBEDDING_DIMENSIONS = test_emb.shape[1]

print(f"\n모델 로드 완료. 임베딩 차원 수: {EMBEDDING_DIMENSIONS}")


In [ ]:
def embed_texts(texts: List[str], input_type: str = "document", show_progress_bar: bool = False) -> List[List[float]]:
    """
    voyage-4-nano로 텍스트 리스트를 로컬에서 임베딩합니다.
    HuggingFace에서 권장하는 encode_query와 encode_document API를 사용합니다.

    Parameters
    ----------
    texts      : 임베딩할 텍스트 목록
    input_type : "document" (저장용) 또는 "query" (검색용)
    show_progress_bar : tqdm 진행 상태 표시 여부
    """
    if input_type == "query":
        # 검색용 쿼리 임베딩 (내부적으로 "Represent the query for retrieving supporting documents: " 접두사 자동 추가)
        embeddings = embed_model.encode_query(
            texts,
            show_progress_bar=show_progress_bar,
        )
    else:
        # 저장용 문서 청크 임베딩 (내부적으로 "Represent the document for retrieval: " 접두사 자동 추가)
        embeddings = embed_model.encode_document(
            texts,
            show_progress_bar=show_progress_bar,
        )
    return embeddings.tolist()


# 빠른 동작 테스트
sample = embed_texts(["MongoDB vector search test"])
print(f"임베딩 벡터 앞 5개 값: {sample[0][:5]}")


In [ ]:
print(f"{len(all_chunks)}개 청크 임베딩 중...")

texts = [c["body"] for c in all_chunks]
embeddings = embed_texts(texts, input_type="document", show_progress_bar=True)

# 각 청크 딕셔너리에 embedding 필드 추가
for chunk_doc, emb in zip(all_chunks, embeddings):
    chunk_doc["embedding"] = emb

print(f"완료! 총 {len(embeddings)}개 벡터 생성됨.")


## Step 5: MongoDB에 데이터 적재 (Ingest)

임베딩이 포함된 청크 문서들을 Local Atlas 컬렉션에 저장합니다.

> 이 셀은 매 실행 시 기존 데이터를 초기화하고 새로 삽입합니다. 반복 실행해도 안전합니다.


In [ ]:
# 중복 방지: 기존 데이터 삭제
collection.delete_many({})
print("기존 데이터 삭제 완료.")

result = collection.insert_many(all_chunks)
print(f"삽입 완료: {len(result.inserted_ids)}개 문서")
print(f"컬렉션 내 총 문서 수: {collection.count_documents({})}개")


## Step 6: 벡터 검색 인덱스 생성

`$vectorSearch`를 사용하려면 MongoDB Atlas 전용 **벡터 인덱스**가 필요합니다.
이 인덱스는 내부적으로 **HNSW(Hierarchical Navigable Small World)** 알고리즘을 사용하여
수백만 개의 벡터 중에서도 빠르게 유사한 문서를 찾아냅니다.

인덱스 설정:
- `path`: 임베딩이 저장된 필드명 (`embedding`)
- `numDimensions`: 임베딩 벡터의 차원 수 (앞 단계에서 자동 감지)
- `similarity`: 유사도 측정 방식
  - **`cosine`**: 벡터 방향의 코사인 각도로 유사도를 측정합니다. `voyage-4-nano`는 L2 정규화된 벡터를 출력하므로 `cosine`과 `dotProduct`는 동일한 결과를 냅니다. 정규화 여부가 불확실할 때도 안전한 기본값이므로 `cosine`을 사용합니다.
  - `dotProduct`: 정규화된 벡터에서 `cosine`과 동일 결과지만 연산이 약간 빠릅니다. 임베딩이 반드시 정규화됨이 보장될 때 사용합니다.
  - `euclidean`: 벡터 간 직선 거리. 의미 검색(semantic search)에는 적합하지 않습니다.

In [ ]:
search_index_model = SearchIndexModel(
    definition={
        "fields": [
            {
                "type": "vector",
                "path": "embedding",
                "numDimensions": EMBEDDING_DIMENSIONS,
                "similarity": "cosine",
            }
        ]
    },
    name=VECTOR_INDEX_NAME,
    type="vectorSearch",
)

# 기존 인덱스가 있으면 삭제 후 재생성 (반복 실행 안전)
existing = list(collection.list_search_indexes(name=VECTOR_INDEX_NAME))
if existing:
    collection.drop_search_index(VECTOR_INDEX_NAME)
    print("기존 인덱스 삭제 중...")
    time.sleep(5)

collection.create_search_index(model=search_index_model)
print(f"인덱스 '{VECTOR_INDEX_NAME}' 생성 요청 완료. READY 상태를 기다립니다...")

In [ ]:
def wait_for_index(col, index_name: str, timeout: int = 180) -> None:
    """인덱스가 READY 상태가 될 때까지 폴링합니다."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        indexes = list(col.list_search_indexes(name=index_name))
        status = indexes[0].get("status", "PENDING") if indexes else "PENDING"
        print(f"  현재 상태: {status}", end="\r")
        if status == "READY":
            print(f"\n인덱스 '{index_name}' 준비 완료!")
            return
        time.sleep(5)
    raise TimeoutError(f"인덱스가 {timeout}초 내에 READY 상태가 되지 않았습니다.")

wait_for_index(collection, VECTOR_INDEX_NAME)

## Step 7: 벡터 검색 (Query & Retrieve)

사용자의 질문을 임베딩하고, `$vectorSearch`로 의미적으로 가장 유사한 청크 K개를 검색합니다.

**파이프라인 단계:**
1. 질문 → `voyage-4-nano` (로컬) → 쿼리 벡터
2. `$vectorSearch`: 벡터 공간에서 코사인 유사도로 상위 K개 검색
3. `$project`: `embedding` 필드 제외, `vectorSearchScore` 포함하여 반환

**`numCandidates`란?**
HNSW는 근사 최근접 이웃(ANN) 알고리즘입니다. `numCandidates`는 최종 `limit`개를 선정하기 전에 후보로 검토할 벡터 수입니다. MongoDB 권장값은 `limit`의 최소 10배이며, 값이 클수록 정확도는 높아지지만 속도는 느려집니다.

In [ ]:
def vector_search(query: str, top_k: int = 5) -> List[Dict]:
    """
    쿼리와 의미적으로 유사한 상위 top_k개의 문서 청크를 반환합니다.
    embedding 필드는 결과에서 제외합니다 (크기가 크므로).
    """
    query_embedding = embed_texts([query], input_type="query")[0]

    pipeline = [
        {
            "$vectorSearch": {
                "index": VECTOR_INDEX_NAME,
                "queryVector": query_embedding,
                "path": "embedding",
                "numCandidates": top_k * 10,  # 더 많은 후보 중에서 상위 K개 선택
                "limit": top_k,
            }
        },
        {
            "$project": {
                "_id": 0,
                "embedding": 0,  # 벡터 제외 (출력 간소화)
                "score": {"$meta": "vectorSearchScore"},
            }
        },
    ]

    return list(collection.aggregate(pipeline))


In [ ]:
# 검색 테스트
results = vector_search("What are best practices for MongoDB backups?")

print(f"검색 결과 {len(results)}개:\n")
for i, doc in enumerate(results, 1):
    score = doc.get("score", 0)
    title = doc.get("title", "N/A")
    preview = doc["body"][:120].replace("\n", " ")
    print(f"[{i}] score={score:.4f}  |  {title}")
    print(f"    {preview}...")
    print()


## Step 8: 대화 기록 저장/불러오기 (Memory)

일반 LLM은 매 질문을 독립적으로 처리합니다.
"방금 뭐라고 했지?"처럼 이전 맥락을 참조하는 질문에 답하려면 **대화 기록(Chat History)** 이 필요합니다.

이 실습에서는 MongoDB의 `chat_history` 컬렉션을 메모리 저장소로 활용합니다.
각 메시지는 `session_id`, `role`, `content`, `timestamp` 필드로 저장되며,
`session_id`로 대화 세션을 구분하므로 여러 사용자의 기록을 동시에 관리할 수 있습니다.

In [ ]:
# session_id 필터 + timestamp 정렬을 하나의 인덱스로 커버
history_coll.create_index([("session_id", 1), ("timestamp", 1)])
print("chat_history 컬렉션 인덱스 생성 완료.")


def store_message(session_id: str, role: str, content: str) -> None:
    """한 건의 채팅 메시지를 MongoDB에 저장합니다."""
    history_coll.insert_one({
        "session_id": session_id,
        "role": role,       # "user" 또는 "assistant"
        "content": content,
        "timestamp": datetime.now(timezone.utc),
    })


def get_history(session_id: str) -> List[Dict[str, str]]:
    """
    특정 세션의 전체 대화 기록을 시간 순으로 반환합니다.
    반환 형식: [{"role": "user"|"assistant", "content": "..."}]
    """
    cursor = history_coll.find(
        {"session_id": session_id},
        {"_id": 0, "role": 1, "content": 1},
    ).sort("timestamp", 1)
    return [{"role": m["role"], "content": m["content"]} for m in cursor]

In [ ]:
# 테스트: 메시지 저장 및 조회
TEST_SESSION = "test-001"
history_coll.delete_many({"session_id": TEST_SESSION})  # 초기화

store_message(TEST_SESSION, "user",      "MongoDB가 뭔가요?")
store_message(TEST_SESSION, "assistant", "MongoDB는 NoSQL 문서 데이터베이스입니다.")

history = get_history(TEST_SESSION)
print(f"{len(history)}개의 메시지 확인:")
for msg in history:
    print(f"  [{msg['role']}] {msg['content']}")

# 테스트 데이터 정리
history_coll.delete_many({"session_id": TEST_SESSION})


## Step 9: 답변 생성 (Gemma 4 via Ollama)

이제 모든 재료가 준비되었습니다. 최종 `generate_answer` 함수는 다음을 한 번에 실행합니다:

1. **벡터 검색** → 관련 문서 청크(Context) 가져오기
2. **메모리 불러오기** → MongoDB에서 이전 대화 기록 가져오기
3. **프롬프트 구성** → Context + 과거 대화 + 현재 질문 조합
4. **Gemma 4 호출** → Ollama OpenAI-compatible API (`/v1/chat/completions`)
5. **저장** → 질문과 답변을 MongoDB에 기록

Ollama는 OpenAI API와 호환되는 엔드포인트를 제공하므로, `openai` 패키지를 그대로 활용합니다.


In [ ]:
from openai import OpenAI

ollama_client = OpenAI(
    base_url=f"{OLLAMA_BASE_URL}/v1",
    api_key="ollama",  # Ollama는 API 키를 사용하지 않지만 필드는 필요
)

# Ollama 연결 및 모델 확인
models = ollama_client.models.list()
available = [m.id for m in models.data]
print("사용 가능한 Ollama 모델:", available)

if OLLAMA_MODEL not in available:
    raise RuntimeError(f"모델 '{OLLAMA_MODEL}'이 Ollama에 없습니다. .env의 OLLAMA_MODEL을 확인하세요.")

print(f"'{OLLAMA_MODEL}' 준비 완료.")

In [ ]:
def generate_answer(session_id: str, user_query: str) -> str:
    """
    RAG + Memory 파이프라인으로 사용자 질문에 답변합니다.

    Parameters
    ----------
    session_id : 대화 세션 ID (사용자 구분용)
    user_query : 사용자의 질문 문자열

    Returns
    -------
    str : LLM이 생성한 답변
    """
    # 1. 벡터 검색: 관련 Context 가져오기
    search_results = vector_search(user_query, top_k=5)
    context = "\n\n".join(doc["body"] for doc in search_results)

    # 2. 시스템 프롬프트 (Context 포함)
    system_content = (
        "You are a helpful assistant that answers questions about MongoDB. "
        "Use the provided context or conversation history to answer the question. "
        "If the answer cannot be found in the context or conversation history, say \"I don't know based on the available documentation.\"\n\n"
        f"Context:\n{context}"
    )

    messages: List[Dict] = [
        {"role": "system", "content": system_content},
    ]

    # 3. 이전 대화 기록(Memory) 추가
    past = get_history(session_id)
    messages.extend(past)

    # 4. 현재 사용자 질문 추가
    messages.append({"role": "user", "content": user_query})

    # 5. Gemma 4 (Ollama) 호출
    response = ollama_client.chat.completions.create(
        model=OLLAMA_MODEL,
        messages=messages,
        temperature=0.1,  # 낮은 온도 = 더 일관적이고 사실에 충실한 답변
    )
    answer = response.choices[0].message.content

    # 6. 대화 기록을 MongoDB에 저장
    store_message(session_id, "user",      user_query)
    store_message(session_id, "assistant", answer)

    return answer

## 실행: RAG 파이프라인 테스트

첫 번째 질문을 실행해 봅니다.
벡터 검색으로 관련 문서를 찾고, Gemma 4가 그것을 바탕으로 답변을 생성합니다.


In [ ]:
MY_SESSION = "session2-demo"

# 이전 실험 기록 초기화 (깨끗한 상태로 시작)
history_coll.delete_many({"session_id": MY_SESSION})

q1 = "What are some best practices for data backups in MongoDB?"
print(f"Q: {q1}\n")
a1 = generate_answer(MY_SESSION, q1)
print(f"A:\n{a1}")


### 메모리 테스트: 이전 질문을 기억하는지 확인

이번 질문은 이전 대화 내용을 참조해야 답할 수 있습니다.
메모리가 없다면 "무엇을 물었는지 모른다"고 답할 것입니다.


In [ ]:
q2 = "Can you summarize what I just asked you about?"
print(f"Q: {q2}\n")
a2 = generate_answer(MY_SESSION, q2)
print(f"A:\n{a2}")


In [ ]:
# MongoDB에 저장된 대화 기록 확인
print("── MongoDB에 저장된 대화 기록 ──")
for msg in get_history(MY_SESSION):
    label = "사용자" if msg["role"] == "user" else "Gemma4"
    print(f"[{label}] {msg['content'][:120]}...")
    print()


## 직접 해보기

아래 질문들로 RAG 파이프라인을 테스트해 보세요.
`MY_SESSION`을 바꾸면 새로운 세션으로 대화를 시작합니다.


In [ ]:
# 원하는 질문으로 바꿔서 실행해 보세요
my_questions = [
    "MongoDB Atlas에서 Database Access History 기능의 보기/조회 제한 사항(Limitations)은 무엇인가요?",
    "What is the recommended threshold and alert condition for monitoring concurrent read or write operations?",
    "PyMongo에서 type과 genre 필드에 복합 인덱스(Compound Index)를 생성하는 파이썬 코드는 무엇인가요?",
    "Which fields in the serverStatus document can be used to monitor concurrent client connections?",
    "What is the exact Atlas CLI command to create a team, e.g., 'atlas teams create'?",
    "Can we rename a team using the Atlas CLI?",
]

for q in my_questions:
    print(f"Q: {q}\n")
    a = generate_answer(MY_SESSION, q)
    print(f"A:\n{a}\n")
    print("─" * 60)
